In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/pima-indians-diabetes-database/diabetes.csv


## This notebook focused on hyperprameter tuning in neural network

In [13]:
df = pd.read_csv('/kaggle/input/pima-indians-diabetes-database/diabetes.csv')

In [14]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [15]:
df.shape

(768, 9)

In [16]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

In [17]:
df.corr()['Outcome']

Pregnancies                 0.221898
Glucose                     0.466581
BloodPressure               0.065068
SkinThickness               0.074752
Insulin                     0.130548
BMI                         0.292695
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [18]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [19]:
# Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [20]:
X = scaler.fit_transform(X)

In [21]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size = 0.2 , random_state = 1)

In [132]:
import tensorflow
from tensorflow import keras 
from keras import Sequential
from keras.layers import Dense,Dropout

In [27]:
model = Sequential()
model.add(Dense(32, activation = 'relu', input_dim=8))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [28]:
model.fit(X_train , y_train , batch_size = 32 , epochs =100, validation_data = (X_test , y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.4911 - loss: 0.7192 - val_accuracy: 0.7013 - val_loss: 0.6539
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6822 - loss: 0.6399 - val_accuracy: 0.7468 - val_loss: 0.6043
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7338 - loss: 0.6046 - val_accuracy: 0.7662 - val_loss: 0.5653
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7567 - loss: 0.5546 - val_accuracy: 0.7727 - val_loss: 0.5359
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7218 - loss: 0.5495 - val_accuracy: 0.7662 - val_loss: 0.5176
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7121 - loss: 0.5461 - val_accuracy: 0.7727 - val_loss: 0.5022
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7464 - loss: 0.5129 - val_accuracy: 0.7792 - val_loss: 0.4892
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7643 - loss: 0.4908 - val_accuracy: 0.7922 - 

SO till now i have to manually determine the no. of hidden layer , activation fun , epochs etc.

In [ ]:
# Automate parameter using keras tuner
# 1. Selecting appropriate optimizer
# 2. Select no of nodes in an leyer
# 3. how to set no. of layers.
# 4. All in one model

In [32]:
import keras_tuner as kt

In [33]:
def build_model(hp):
    model = Sequential()
    model.add(Dense(32,activation='relu',input_dim=8))
    model.add(Dense(1,activation='sigmoid'))

    optimizer = hp.Choice('optimizer',['adam','sgd','rmsprop','adadelta'])
    model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])

    return model



In [34]:
tuner= kt.RandomSearch(build_model ,
                      objective='val_accuracy',
                      max_trials=5)

In [36]:
tuner.search(X_train , y_train , epochs=5, validation_data = (X_test , y_test))


In [38]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [41]:
model = tuner.get_best_models(num_models=1)[0]

In [42]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [45]:
tuner.results_summary()

Results summary
Results in ./untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 0 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.7662337422370911

Trial 1 summary
Hyperparameters:
optimizer: adam
Score: 0.7662337422370911

Trial 3 summary
Hyperparameters:
optimizer: sgd
Score: 0.6948052048683167

Trial 2 summary
Hyperparameters:
optimizer: adadelta
Score: 0.4350649416446686


In [51]:
model.fit(X_train , y_train , batch_size=32 ,epochs=100, initial_epoch = 6 , validation_data = (X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8234 - loss: 0.3890 - val_accuracy: 0.8052 - val_loss: 0.4618
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8008 - loss: 0.4043 - val_accuracy: 0.8117 - val_loss: 0.4609
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7962 - loss: 0.4309 - val_accuracy: 0.8052 - val_loss: 0.4605
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7945 - loss: 0.4406 - val_accuracy: 0.8052 - val_loss: 0.4618
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7941 - loss: 0.4283 - val_accuracy: 0.8052 - val_loss: 0.4621
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8043 - loss: 0.4202 - val_accuracy: 0.8052 - val_loss: 0.4616
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8098 - loss: 0.4082 - val_accuracy: 0.8052 - val_loss: 0.4615
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7881 - loss: 0.4246 - val_accuracy: 0.805

In [91]:
import tensorflow as tf
def build_model(hp):
    model = Sequential()

    model.add(tf.keras.Input(shape=(8,)))

    units = hp.Int('units', min_value=8, max_value=128, step=8)

    model.add(Dense(units=units, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    optimizer = tf.keras.optimizers.RMSprop()

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


In [92]:
tuners = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    overwrite=True,   # IMPORTANT
    directory='mydir',
    project_name='helloworld_v2'
)


max_trials = k → try k random models → keep the best.

In [93]:
tuners.search(X_train , y_train , epochs = 5 , validation_data=(X_test, y_test))

Trial 5 Complete [00h 00m 02s]
val_accuracy: 0.7727272510528564

Best val_accuracy So Far: 0.798701286315918
Total elapsed time: 00h 00m 11s


In [94]:
tuners.search_space_summary()


Search space summary
Default search space size: 1
units (Int)
{'default': None, 'conditions': [], 'min_value': 8, 'max_value': 128, 'step': 8, 'sampling': 'linear'}


In [95]:
tuners.get_best_hyperparameters()[0].values


{'units': 104}

In [97]:
model = tuners.get_best_models(num_models=1)[0]

In [99]:
model.fit(X_train , y_train , batch_size=32 , epochs=100 , initial_epoch=6 , validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7705 - loss: 0.5100 - val_accuracy: 0.7922 - val_loss: 0.4885
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7512 - loss: 0.5027 - val_accuracy: 0.7857 - val_loss: 0.4729
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7786 - loss: 0.4738 - val_accuracy: 0.7857 - val_loss: 0.4656
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8005 - loss: 0.4441 - val_accuracy: 0.7922 - val_loss: 0.4593
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7945 - loss: 0.4448 - val_accuracy: 0.8052 - val_loss: 0.4585
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7804 - loss: 0.4417 - val_accuracy: 0.7987 - val_loss: 0.4590
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7651 - loss: 0.4531 - val_accuracy: 0.7987 - val_loss: 0.4568
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7530 - loss: 0.4695 - val_accuracy: 0.79

In [115]:
# How to select Number of layers
def build_model(hp):
    model = Sequential()

    #Input layer
    model.add(Dense(104 , activation='relu', input_dim = 8))
    for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
        model.add(Dense(104 , activation='relu'))

    #output
    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer='rmsprop', loss='binary_crossentropy',metrics=['accuracy'])

    return model

tuner = kt.RandomSearch(build_model,
                       objective='val_accuracy',
                       max_trials = 3,
                       directory = 'mydir',
                       project_name='num_layersv2')

Reloading Tuner from mydir/num_layersv2/tuner0.json


In [116]:
tuner.search(X_train , y_train , epochs=5, validation_data=(X_test, y_test))

Trial 10 Complete [00h 00m 03s]
val_accuracy: 0.798701286315918

Best val_accuracy So Far: 0.8116883039474487
Total elapsed time: 00h 08m 30s


In [117]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 9}

In [118]:
tuner.search_space_summary()

Search space summary
Default search space size: 1
num_layers (Int)
{'default': None, 'conditions': [], 'min_value': 1, 'max_value': 10, 'step': 1, 'sampling': 'linear'}


**When using Keras Tuner RandomSearch, max_trials = k means it randomly selects k values from the defined hyperparameter range, builds k different models, and returns the best one among those k trials — not the global best.**

In [139]:
#All models
def build_model(hp):
    model = Sequential()
    counter = 0

    for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
        if counter == 0 :
            model.add(
                Dense(
                    hp.Int('units'+str(i),min_value=8,max_value=128,step=8),
                    activation=hp.Choice('activation'+str(i),values=['relu','tanh','sigmoid']),
                    input_dim=8
                )
            )
            model.add(Dropout(hp.Choice('dropout'+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
        else:
            model.add(
                Dense(
                    hp.Int('units'+str(i),min_value=8,max_value=128,step=8),
                    activation=hp.Choice('activation'+str(i),values=['relu','tanh','sigmoid']),
                )
            )
            model.add(Dropout(hp.Choice('dropout'+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))

        counter += 1

    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer=hp.Choice('optimizer',values=['rmsprop','adam','sgd','nadam','adadelta']),
                 loss='binary_crossentropy',
                 metrics=['accuracy'])

    return model
            

In [140]:
tuner = kt.RandomSearch(build_model,
                       objective='val_accuracy',
                       max_trials=3,
                       directory='mydir',
                       project_name='final_dropout')

In [141]:
tuner.search(X_train, y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.6558441519737244

Best val_accuracy So Far: 0.7662337422370911
Total elapsed time: 00h 00m 12s


In [142]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 3,
 'units0': 128,
 'activation0': 'tanh',
 'dropout0': 0.5,
 'optimizer': 'rmsprop',
 'units1': 88,
 'activation1': 'sigmoid',
 'dropout1': 0.2,
 'units2': 40,
 'activation2': 'tanh',
 'dropout2': 0.9,
 'units3': 56,
 'activation3': 'sigmoid',
 'dropout3': 0.1,
 'units4': 16,
 'activation4': 'sigmoid',
 'dropout4': 0.1,
 'units5': 128,
 'activation5': 'relu',
 'dropout5': 0.5,
 'units6': 8,
 'activation6': 'tanh',
 'dropout6': 0.5,
 'units7': 88,
 'activation7': 'sigmoid',
 'dropout7': 0.9,
 'units8': 80,
 'activation8': 'relu',
 'dropout8': 0.9,
 'units9': 88,
 'activation9': 'sigmoid',
 'dropout9': 0.6}

In [143]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [144]:
model.fit(X_train,y_train,epochs=200,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6859 - loss: 0.6630 - val_accuracy: 0.7662 - val_loss: 0.5030
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6740 - loss: 0.6406 - val_accuracy: 0.7662 - val_loss: 0.4864
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6828 - loss: 0.6365 - val_accuracy: 0.7727 - val_loss: 0.4819
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6945 - loss: 0.6189 - val_accuracy: 0.7727 - val_loss: 0.4868
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7320 - loss: 0.5770 - val_accuracy: 0.7597 - val_loss: 0.4712
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7330 - loss: 0.5826 - val_accuracy: 0.7727 - val_loss: 0.4742
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7135 - loss: 0.5988 - val_accuracy: 0.7662 - val_loss: 0.4691
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7071 - loss: 0.5843 - val_accuracy: 0.77